In [1]:
import os
import sqlite3

import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

from utilities import get_path, read_data_from_database, save_dataframe_to_sqlite

In [2]:
pd.options.display.max_columns = None  # Remove "dots" from display when printing dataframes

In [3]:
PATH = get_path(1)
DATABASE_FOLDER = PATH + 'data/preprocessing/clear_data.db'

In [4]:
df_train = read_data_from_database(DATABASE_FOLDER, 'train_table')
df_test = read_data_from_database(DATABASE_FOLDER, 'test_table')

df_train.head()

,ID,Edad,Tipo_Trabajo,Estado_Civil,Educacion,mora,Vivienda,Consumo,Contacto,Mes,Dia,Campana,Dias_Ultima_Camp,No_Contactos,Resultado_Anterior,emp_var_rate,cons_price_idx,cons_conf_idx,euribor3m,nr_employed,y
0,1,57,servicios,casado,bachillerato,NaN,0.0,0.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0
1,2,37,servicios,casado,bachillerato,0.0,1.0,0.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0
2,3,40,administrador negocio,casado,primaria,0.0,0.0,0.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0
3,4,56,servicios,casado,bachillerato,0.0,0.0,1.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0
4,7,25,servicios,soltero,bachillerato,0.0,1.0,0.0,telefono fijo,may,lun,1,-1,0,sin contacto,1.1,93994.0,-36.4,4857.0,5191,0


In [5]:
df_train.isnull().sum()

ID                       0
Edad                     0
Tipo_Trabajo           165
Estado_Civil            39
Educacion              958
mora                  4766
Vivienda               553
Consumo                553
Contacto                 0
Mes                      0
Dia                      0
Campana                  0
Dias_Ultima_Camp         0
No_Contactos             0
Resultado_Anterior       0
emp_var_rate             0
cons_price_idx           0
cons_conf_idx            0
euribor3m                0
nr_employed              0
y                        0
dtype: int64

# Missing values

In [6]:
# Simple imputation
cols_inputation = ['Estado_Civil', 'Tipo_Trabajo', 'Educacion']

for column in cols_inputation:
    df_train[column].fillna(df_train[column].mode()[0], inplace=True)
    df_test[column].fillna(df_test[column].mode()[0], inplace=True)

In [7]:
# KNN imputattion
cols_inputation = ['mora', 'Vivienda', 'Consumo']

for column in cols_inputation:
       number_neighbors = len(df_train[column].dropna().unique())
       imputer = KNNImputer(n_neighbors=number_neighbors, weights='uniform')

       print(f"Valores perdidos en {column} train: {str(df_train[column].isnull().sum())}")
       print(f"Valores perdidos en {column} test: {str(df_test[column].isnull().sum())}")

       imputer.fit(df_train[[column]])
       df_train[column] = imputer.transform(df_train[[column]]).ravel().round()
       df_test[column] = imputer.transform(df_test[[column]]).ravel().round()  # Replace in test data

       print(f"Valores perdidos en {column} train: {str(df_train[column].isnull().sum())}")
       print(f"Valores perdidos en {column} test: {str(df_test[column].isnull().sum())}\n")

Valores perdidos en mora train: 4766
Valores perdidos en mora test: 2058
Valores perdidos en mora train: 0
Valores perdidos en mora test: 0

Valores perdidos en Vivienda train: 553
Valores perdidos en Vivienda test: 234
Valores perdidos en Vivienda train: 0
Valores perdidos en Vivienda test: 0

Valores perdidos en Consumo train: 553
Valores perdidos en Consumo test: 234
Valores perdidos en Consumo train: 0
Valores perdidos en Consumo test: 0



In [8]:
df_train.isnull().sum()

ID                    0
Edad                  0
Tipo_Trabajo          0
Estado_Civil          0
Educacion             0
mora                  0
Vivienda              0
Consumo               0
Contacto              0
Mes                   0
Dia                   0
Campana               0
Dias_Ultima_Camp      0
No_Contactos          0
Resultado_Anterior    0
emp_var_rate          0
cons_price_idx        0
cons_conf_idx         0
euribor3m             0
nr_employed           0
y                     0
dtype: int64

# Save

In [9]:
# Load to DataBase
# Create a connection to the SQLite database
conn = sqlite3.connect(PATH + 'data/preprocessing/BANCO_BOGOTA.db')

# Assume df_train_train and df_train_test are your existing DataFrames
dataframes = {'preproc_train_data': df_train, 'preproc_test_data': df_test}

try:
    # Create a connection to the SQLite database
    conn = sqlite3.connect(os.path.join(PATH, 'data', 'preprocessing', 'prep_data.db'))

    # Iterate over the DataFrames and their corresponding table names
    for table_name, df in dataframes.items():
        # Save the DataFrame to the SQLite database
        df.to_sql(table_name, conn, index=False, if_exists='replace')

    # Commit changes
    conn.commit()
    print('Data loaded successfully to the database.')
except Exception as e:
    print(f'An error occurred while loading data to the database: {str(e)}')
    if conn is not None:
        conn.rollback()
finally:
    # Close connection
    if conn is not None:
        conn.close()
        print('Database connection closed.')

Data loaded successfully to the database.
Database connection closed.
